# Chapter 21 — Nonparametric Statistics

## Learning Objectives
- Explain the basic idea of nonparametric methods and why ranks are useful
- Distinguish parametric and nonparametric approaches
- Perform Mann–Whitney U and Wilcoxon signed-rank tests
- Calculate and interpret Spearman rank correlation
- Recognize that nonparametric methods still have assumptions
- Choose methods based on the research question and data structure

## 1. What Does Nonparametric Mean?

Nonparametric methods often rely less directly on a specific parametric distributional form. They are especially useful when ranks, ordinal measurements, distributional comparisons, or monotonic relationships match the analytical question.

> **Nonparametric does not mean assumption-free.**

Independence, pairing, measurement scale, symmetry, and the precise null hypothesis can still matter.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/.")

ROOT = find_project_root(Path.cwd())
df = pd.read_csv(ROOT / "data" / "21_nonparametric_statistics.csv")
df.head()

## 2. Dataset

The synthetic dataset supports three questions:

| Variable | Description |
|---|---|
| `group` | Independent group A or B |
| `completion_time_minutes` | Task completion time |
| `score_before`, `score_after` | Paired scores |
| `satisfaction_level` | Ordinal level, 1–5 |
| `response_time_minutes` | Quantitative response time |

All observations are fictional.

In [ ]:
df.describe(include='all')

## 3. Why Ranks?

Several nonparametric methods replace raw values with their ranks. For

$$
12,\ 30,\ 18,\ 25
$$

the ordered values are

$$
12 < 18 < 25 < 30
$$

so their ranks are 1, 4, 2, and 3 in the original order. Ranks preserve ordering while reducing dependence on exact numerical distances.

In [ ]:
example = pd.Series([12, 30, 18, 25])
pd.DataFrame({"value": example, "rank": example.rank()})

### Ties

Tied observations commonly receive the average of the ranks they would occupy. For values 10, 20, 20, 40, the two 20s each receive rank

$$
\frac{2+3}{2}=2.5
$$

In [ ]:
tied = pd.Series([10, 20, 20, 40])
pd.DataFrame({"value": tied, "rank": tied.rank(method="average")})

## 4. Parametric vs Rank-Based Alternatives

| Question | Common parametric method | Common rank-based alternative |
|---|---|---|
| Two independent groups | Independent t-test | Mann–Whitney U |
| Two paired measurements | Paired t-test | Wilcoxon signed-rank |
| Association | Pearson correlation | Spearman rank correlation |

These methods do not necessarily test exactly the same feature. Selection should follow the research question rather than a mechanical normality rule.

# Part A — Mann–Whitney U Test

## 5. Two Independent Groups

We compare task completion times for independent Groups A and B. First inspect the distributions.

In [ ]:
a = df.loc[df["group"]=="A", "completion_time_minutes"]
b = df.loc[df["group"]=="B", "completion_time_minutes"]
pd.DataFrame({"A": a.describe(), "B": b.describe()})

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot([a,b], tick_labels=["Group A","Group B"])
plt.ylabel("Completion Time (minutes)")
plt.title("Completion Time by Group")
plt.grid(axis="y", alpha=.25)
plt.show()

## 6. Mann–Whitney U

The Mann–Whitney U test compares two independent groups using ranks. A broad two-sided null hypothesis is

$$
H_0:\ \text{the two groups have the same distribution}
$$

Under additional conditions, such as similarly shaped distributions differing mainly by location, it is often interpreted as a location comparison. It should not automatically be called simply a "test of medians."

In [ ]:
u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
print("U statistic:", u)
print("p-value:", p)

A small p-value provides evidence against the null hypothesis. A large p-value does not prove equality, and statistical significance does not measure practical importance.

## 7. Inspect the Ranks

In [ ]:
ranked = df[["group","completion_time_minutes"]].copy()
ranked["rank"] = ranked["completion_time_minutes"].rank(method="average")
ranked.groupby("group")["rank"].agg(["count","mean","sum"])

If one group tends to contain larger observations, it also tends to receive larger ranks. The U statistic summarizes this ordering information.

# Part B — Wilcoxon Signed-Rank Test

## 8. Paired Measurements

Each employee has a before and after score. Because both measurements belong to the same employee, the observations are paired.

In [ ]:
df["score_difference"] = df["score_after"] - df["score_before"]
df[["employee_id","score_before","score_after","score_difference"]].head(10)

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["score_difference"], bins=12, edgecolor="black")
plt.xlabel("After - Before")
plt.ylabel("Frequency")
plt.title("Distribution of Paired Differences")
plt.show()

## 9. Wilcoxon Signed-Rank Test

A common two-sided formulation is

$$
H_0:\ \text{the distribution of paired differences is symmetric about }0
$$

The method uses both the signs of nonzero differences and the ranks of their absolute magnitudes.

In [ ]:
w, p = stats.wilcoxon(df["score_after"], df["score_before"], alternative="two-sided")
print("Wilcoxon statistic:", w)
print("p-value:", p)

For the usual location interpretation, symmetry of paired differences matters. Wilcoxon signed-rank is therefore not merely "a paired t-test without normality."

## 10. How Signed Ranks Work

In [ ]:
demo = df[["employee_id","score_difference"]].copy()
demo = demo[demo["score_difference"] != 0].copy()
demo["absolute_difference"] = demo["score_difference"].abs()
demo["absolute_rank"] = demo["absolute_difference"].rank(method="average")
demo["signed_rank"] = np.sign(demo["score_difference"]) * demo["absolute_rank"]
demo.head(15)

In [ ]:
print("Positive rank sum:", demo.loc[demo["signed_rank"]>0,"absolute_rank"].sum())
print("Negative rank sum:", demo.loc[demo["signed_rank"]<0,"absolute_rank"].sum())

# Part C — Spearman Rank Correlation

## 11. Monotonic Association

We examine satisfaction level (ordinal, 1–5) and response time. Pearson correlation targets linear association; Spearman correlation targets monotonic rank association.

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["satisfaction_level"], df["response_time_minutes"], alpha=.7)
plt.xlabel("Satisfaction Level")
plt.ylabel("Response Time (minutes)")
plt.title("Satisfaction vs Response Time")
plt.grid(alpha=.25)
plt.show()

## 12. Spearman's Rank Correlation

Spearman's coefficient is often denoted by $\rho_s$ and satisfies

$$
-1 \le \rho_s \le 1
$$

Values near +1 indicate strong increasing monotonic association; values near -1 indicate strong decreasing monotonic association.

In [ ]:
rho, p = stats.spearmanr(df["satisfaction_level"], df["response_time_minutes"])
print("Spearman correlation:", rho)
print("p-value:", p)

## 13. Verify Spearman Using Ranks

Spearman correlation can be understood as Pearson correlation applied to ranked variables, with ties handled by the ranking procedure.

In [ ]:
x_rank = df["satisfaction_level"].rank(method="average")
y_rank = df["response_time_minutes"].rank(method="average")
manual = x_rank.corr(y_rank)
print("Correlation between ranks:", manual)
print("SciPy Spearman:", rho)
print("Same result:", np.isclose(manual, rho))

## 14. Pearson vs Spearman

In [ ]:
pearson_r, pearson_p = stats.pearsonr(df["satisfaction_level"], df["response_time_minutes"])
pd.DataFrame({
    "method":["Pearson","Spearman"],
    "correlation":[pearson_r,rho],
    "p_value":[pearson_p,p],
})

Pearson asks about linear association; Spearman asks about monotonic rank association. Spearman is not automatically better merely because a variable is skewed or ordinal—the target question matters.

## 15. Common Misconceptions

1. **"Nonparametric methods have no assumptions."** False.
2. **"Mann–Whitney U is always a test of medians."** Not without additional conditions.
3. **"Wilcoxon is just a paired t-test without normality."** Too simplistic; signed-rank has its own conditions.
4. **"Use Spearman whenever normality fails."** Pearson and Spearman target different forms of association.

## 16. Choosing a Method

Start with the analytical target:

- Independent groups + mean difference → an independent t-test may fit the question.
- Independent groups + rank/distribution comparison → Mann–Whitney U may fit.
- Paired mean difference → paired t-test may fit.
- Paired rank-based location comparison with suitable conditions → Wilcoxon signed-rank may fit.
- Linear association → Pearson.
- Monotonic rank association → Spearman.

The goal is not to eliminate assumptions. It is to use assumptions and a target quantity that make sense for the problem.

## 17. Practical Workflow

$$
\text{Research Question}
\rightarrow
\text{Data Structure}
\rightarrow
\text{Measurement Scale / Distribution}
\rightarrow
\text{Target Quantity}
\rightarrow
\text{Method}
\rightarrow
\text{Statistic and p-value}
\rightarrow
\text{Interpretation and Limitations}
$$

## 18. Exercises

1. Rank the values 8, 12, 12, 20, 30 and explain ties.
2. Compare Groups A and B visually and with Mann–Whitney U.
3. Explain why Mann–Whitney U is not automatically a median test.
4. Inspect paired score differences and run Wilcoxon signed-rank.
5. Explain why signed-rank uses both signs and magnitudes.
6. Calculate and interpret Spearman correlation.
7. Verify Spearman manually using ranks.
8. Compare Pearson and Spearman on the same variables.
9. Give a situation where a t-test better matches the target question.
10. Explain why "nonparametric means assumption-free" is incorrect.

## 19. Summary

This chapter introduced three rank-based methods:

- **Mann–Whitney U:** two independent groups
- **Wilcoxon signed-rank:** paired observations
- **Spearman rank correlation:** monotonic association

The central lesson is:

> **Nonparametric does not mean assumption-free.**

Method selection should begin with the research question and data structure—not a mechanical rule such as "if normality fails, use a nonparametric test."

### Next Chapter

**Chapter 22 — Bootstrap**

We will repeatedly resample observed data to approximate sampling uncertainty computationally.